# Controlled changes on the AI-only track

The judge ablations of [two_track_figure.ipynb](two_track_figure.ipynb), run on pairs where both
ideas are generated with the same prompt: gpt-5.4 (positive) vs. gpt-5.1 (negative).

The label assumes the stronger model's idea is the more novel one. So next to accuracy we
report the **flip rate**: the share of instances whose verdict changes under an ablation,
which needs no label.

Reads the `merge_ai_gpt.yaml` merge.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from IPython.display import Image, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src").is_dir() and (p / "output").is_dir())
sys.path.insert(0, str(ROOT / "src"))

from novelty_eval.figures.common import FONT_STACK, PDF_RCPARAMS, TwoTrackFigure

plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": FONT_STACK,
                     **PDF_RCPARAMS})
print(ROOT)

In [ ]:
MERGE = "merge_ai_gpt"
TRACK = "pairwise-ai-gpt"
DATA = "data/ai-only/gpt/pairwise.yaml"
ROWS = [
        ("ai_researcher_base", 'P3 · "Which was\njudged novel?"'),
        ("ai_researcher_no_review", 'P4 · "Which is novel?"'),
        ("ai_researcher_comparative", 'P5 · "Which was judged\n more novel?"')]
MODELS = ["claude-sonnet-4-5", "claude-opus-4-5", "claude-opus-4-6",
          "gpt-5.1", "gpt-5.2", "gpt-5.4"]

FIG = TwoTrackFigure(root=ROOT, figure="ai-only",
                     tracks=[("AI-only (gpt-5.4 vs. gpt-5.1)", MERGE, "")],
                     rows=ROWS, models=MODELS,
                     metrics={"pairwise": "pairwise_accuracy_strict"},
                     row_groups=[("Judge prompt", [name for name, _ in ROWS[:5]])],
                     row_axis_label="Change", baseline_label="Reference")

## Accuracy

In [ ]:
print(FIG.table("pairwise").round(1).to_string(index=False))
png = FIG.draw("pairwise")
print(png.relative_to(ROOT), "(+ .pdf)")
display(Image(filename=str(png)))

## Flip rate

% of instances whose verdict (idea 0, idea 1 or tie) differs from the same judge's
verdict under `current`. Averaged over runs when an ablation has several.

In [ ]:
ARTIFACTS = ROOT / "output/ablation_sweeps" / MERGE / "artifacts"


def verdicts(ablation, model):
    """One {instance id: verdict} dict per run, from the newest artifact dir with scores."""
    files = sorted(ARTIFACTS.glob(f"{TRACK}_{ablation}-{model}_*/run_*/scores.json"))
    newest = [f for f in files if f.parents[1] == files[-1].parents[1]]
    return [{k: v["comparisons"][0]["winner"] for k, v in json.loads(f.read_text()).items()}
            for f in newest]


recs = []
for model in MODELS:
    base = verdicts("current", model)
    for name, label in ROWS:
        for run in verdicts(name, model):
            ids = base[0].keys() & run.keys()
            recs.append({"ablation": label, "judge": model,
                         "flip %": 100 * sum(base[0][i] != run[i] for i in ids) / len(ids)})

FLIPS = (pd.DataFrame(recs).groupby(["ablation", "judge"])["flip %"].mean().unstack()
         .reindex(index=[label for _, label in ROWS], columns=MODELS))
print(FLIPS.round(1).to_string())

### Flip breakdown

Number of instances per kind of flip, reference → ablation: ✓ correct, ✗ wrong, tie.
`%` is the total over all instances. Averaged over runs when an ablation has several.
Shown against `current`, and against P3 for the other minimal prompts.

In [ ]:
GOLD = {str(k): v["expected_winners"][0] for k, v in yaml.safe_load((ROOT / DATA).read_text()).items()}
TYPES = ["✓→✗", "✗→✓", "✓→tie", "✗→tie", "tie→✓", "tie→✗"]


def mark(i, verdict):
    return "tie" if verdict == 2 else "✓" if verdict == GOLD[i] else "✗"


def breakdown(ref, rows):
    """Instances per kind of flip from `ref` to each of `rows`, per judge."""
    recs = []
    for model in MODELS:
        base = verdicts(ref, model)[0]
        for name, label in rows:
            runs = verdicts(name, model)
            for run in runs:
                for i in base.keys() & run.keys():
                    if base[i] != run[i]:
                        recs.append({"ablation": label, "judge": model, "n": 1 / len(runs),
                                     "flip": f"{mark(i, base[i])}→{mark(i, run[i])}"})
    table = (pd.DataFrame(recs)
             .pivot_table(index=["ablation", "judge"], columns="flip", values="n", aggfunc="sum")
             .reindex(index=pd.MultiIndex.from_product([[label for _, label in rows], MODELS]),
                      columns=TYPES)
             .fillna(0))
    table["total"] = table.sum(axis=1)
    table["%"] = 100 * table["total"] / len(GOLD)
    return table


print("Reference: current")
print(breakdown("current", ROWS).round(1).to_string(), "\n")
print("Reference: P3")
print(breakdown("ai_researcher_base", [r for r in ROWS if r[0] != "ai_researcher_base"])
      .round(1).to_string())